# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR⁲ dataset using the `mlcroissant` library. You will learn how to programmatically access record sets, fields, and columns by their `@id` and perform initial exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install `mlcroissant` if it is not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`. All exploration uses Croissant schema `@id` for precision and reproducibility.

In [ ]:
# List all top-level information about record sets in the dataset
# Note: Use `dataset.record_sets` to access record set objects and their IDs.

print("Available record sets and their `@id`s:")
for record_set in dataset.record_sets:
    print(f"- {record_set.id} (name: {record_set.name})")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for fld in record_set.fields:
            print(f"    - {fld.id} (name: {fld.name})")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for col in record_set.columns:
            print(f"    - {col.id} (name: {col.name})")
    print()

## 3. Data Extraction
Extract data from available record sets into pandas DataFrames. Reference all record sets and fields using their Croissant `@id`.

_If there is more than one record set, we'll load each._

In [ ]:
# Build a list of record set @ids and load their data as DataFrames
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"First 3 records:\n{df.head(3)}\n")

# Choose the main record set for demonstration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set chosen for analysis: {main_record_set_id}")
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze data from the main record set. We'll:
- Select a numeric field (by `@id`)
- Filter records by threshold
- Normalize the field
- Optionally, group by a categorical field.

_Remember to always use fields and columns by their Croissant `@id` for programmatic clarity._

In [ ]:
# Inspect available columns in the main record set
df = dataframes[main_record_set_id]
print(f"Columns in {main_record_set_id}: {list(df.columns)}\n")

# Try to choose a numeric field by inspecting the DataFrame
numeric_candidates = df.select_dtypes(include=["number"]).columns.tolist()
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Try to coerce columns to numeric just in case
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors="coerce")
        n_na = coerced.isna().sum()
        if n_na < len(df):
            numeric_field_id = col
            df[numeric_field_id] = coerced
            break
    else:
        numeric_field_id = None

if numeric_field_id is None:
    raise ValueError("No numeric field detected for EDA.")

print(f"Using numeric field `@id`: {numeric_field_id}")

# Filtering: values greater than a threshold (use 10 as generic example)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a suitable grouping (categorical) field by inspecting object columns
object_candidates = df.select_dtypes(include=["object"]).columns.tolist()
group_field_id = None
for col in object_candidates:
    unique_vals = df[col].nunique()
    if unique_vals > 1 and unique_vals < len(df) // 2:
        group_field_id = col
        break

if group_field_id:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nGrouped mean of {numeric_field_id} by `{group_field_id}`:")
    print(grouped.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and (if possible) the grouped means by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a categorical group is available, plot group means
if group_field_id:
    means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    sns.barplot(x=means.index.astype(str), y=means.values)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated a full workflow to load, overview, extract, and analyze the FAIR² dataset, always referencing record sets, fields, and columns by their schema `@id`. You can now extend this template for more sophisticated analyses, modeling, or reporting as suits your research.